# Preprocessing

## Where this data comes from (simple English)

The table path comes from **`config.yaml`** (`data.customer_training_csv`). By default this is the augmented file that adds a **public uninsured-rate proxy** from scraped Wikipedia state tables (see `notebooks/03_scraped_data_retrain_and_metrics.ipynb`).

- It is **synthetic** (made up by a small random-data recipe), **not** copied from a real company.
- It is meant to **look like** a simple customer snapshot: demographics, rough shopping behavior, engagement signals, and a **churn** label (did they leave?).
- The file was **exported once** from that recipe using the seed in `config.yaml`, so the CSV in the repo is **fixed** and shareable.

### Columns that start with `_` (underscore)

Two columns (`_acquisition_prob`, `_monthly_spend_intensity`) are **only for the teaching simulation** later (to mimic “would this prospect convert?” without cheating).

**Do not treat `_...` columns as real inputs you would normally have at training time.** In this demo they exist so we can compare smart targeting vs random targeting fairly.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

# Plots in notebooks should render inline.
%matplotlib inline

ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists():
    # If launched from notebooks/, walk up to project root.
    ROOT = ROOT.parent

CFG = yaml.safe_load((ROOT / "config.yaml").read_text(encoding="utf-8"))
SEED = int(CFG["project"]["random_seed"])

DATA_REL = str(CFG.get("data", {}).get("customer_training_csv", "data/customers.csv"))
DATA_PATH = Path(DATA_REL) if Path(DATA_REL).is_absolute() else (ROOT / DATA_REL)
df = pd.read_csv(DATA_PATH)
df.head()

## Quick checks

Below: table shape, missing values, churn rate, and a couple of simple plots.

In [ ]:
print("rows, cols:", df.shape)
print("missing per column:\n", df.isna().sum())
print("churn rate:", float(df["churned"].mean()))

_profile_cols = [
    "age",
    "tenure_months",
    "num_orders",
    "total_spend",
    "avg_order_value",
    "days_since_last_order",
    "email_opens_30d",
    "app_sessions_30d",
]
if "public_uninsured_rate_proxy" in df.columns:
    _profile_cols.append("public_uninsured_rate_proxy")

summary = df[_profile_cols].describe().T
summary

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.countplot(data=df, x="region", ax=axes[0])
axes[0].set_title("Customers by region")

sns.histplot(data=df, x="total_spend", bins=40, kde=True, ax=axes[1])
axes[1].set_title("Total spend distribution")

plt.tight_layout()
plt.show()

## Exploratory analysis (before preprocessing)

Visual relationships between numeric signals, churn, and the public health proxy (when present).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

heat_cols = [c for c in _profile_cols if c in df.columns]
if len(heat_cols) >= 3:
    plt.figure(figsize=(8, 6))
    corr = df[heat_cols + ["churned"]].corr(numeric_only=True)
    sns.heatmap(corr, annot=False, cmap="vlag", center=0)
    plt.title("Correlation heatmap (numeric factors + churn)")
    plt.tight_layout()
    plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
churn_by_region = df.groupby("region")["churned"].mean().reset_index()
sns.barplot(data=churn_by_region, x="region", y="churned", ax=axes[0])
axes[0].set_title("Churn rate by region")
axes[0].set_ylabel("P(churned)")

if "public_uninsured_rate_proxy" in df.columns:
    px = df.groupby("region")["public_uninsured_rate_proxy"].first().reset_index()
    sns.barplot(data=px, x="region", y="public_uninsured_rate_proxy", ax=axes[1])
    axes[1].set_title("Public uninsured proxy by region (from scraped wiki)")
else:
    axes[1].set_visible(False)

plt.tight_layout()
plt.show()


## Build the preprocessing recipe and the train/validation/test split

This matches the earlier Python modules:

- numeric columns: fill missing values with the median, then scale
- `region`: fill missing with the most common value, then one-hot encode

The split is **random with a fixed seed** (because this synthetic table has no real calendar order). If you ever use real event data, you would usually split by **time** instead.

In [ ]:

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import List, Tuple

import numpy as np
import pandas as pd
import yaml
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUMERIC_FEATURES: List[str] = [
    "age",
    "tenure_months",
    "num_orders",
    "total_spend",
    "avg_order_value",
    "days_since_last_order",
    "email_opens_30d",
    "app_sessions_30d",
    "public_uninsured_rate_proxy",
]
CATEGORICAL_FEATURES: List[str] = ["region"]
TARGET_COLUMN = "churned"
INTERNAL_COLUMNS: Tuple[str, ...] = ("_acquisition_prob", "_monthly_spend_intensity")


@dataclass
class TrainValTestSplit:
    X_train: pd.DataFrame
    X_val: pd.DataFrame
    X_test: pd.DataFrame
    y_train: np.ndarray
    y_val: np.ndarray
    y_test: np.ndarray
    acquisition_prob_train: np.ndarray
    acquisition_prob_val: np.ndarray
    acquisition_prob_test: np.ndarray


def make_preprocessor() -> ColumnTransformer:
    numeric_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )
    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, NUMERIC_FEATURES),
            ("cat", categorical_pipe, CATEGORICAL_FEATURES),
        ]
    )


def temporal_or_random_split(
    df: pd.DataFrame,
    train_fraction: float,
    val_fraction: float,
    random_seed: int,
) -> TrainValTestSplit:
    if not 0 < train_fraction < 1:
        raise ValueError("train_fraction must be in (0,1)")
    if not 0 < val_fraction < 1:
        raise ValueError("val_fraction must be in (0,1)")
    if train_fraction + val_fraction >= 1:
        raise ValueError("train_fraction + val_fraction must be < 1")

    work = df.reset_index(drop=True)
    idx = np.arange(len(work))
    rng = np.random.default_rng(random_seed)
    rng.shuffle(idx)

    n = len(idx)
    n_train = int(n * train_fraction)
    n_val = int(n * val_fraction)

    train_idx = idx[:n_train]
    val_idx = idx[n_train : n_train + n_val]
    test_idx = idx[n_train + n_val :]

    def _pack(indices: np.ndarray) -> Tuple[pd.DataFrame, np.ndarray, np.ndarray]:
        part = work.iloc[indices]
        y = part[TARGET_COLUMN].to_numpy()
        latent = part["_acquisition_prob"].to_numpy()
        drop_cols = list(INTERNAL_COLUMNS) + [TARGET_COLUMN]
        if "customer_id" in part.columns:
            drop_cols.append("customer_id")
        X = part.drop(columns=drop_cols)
        return X, y, latent

    X_train, y_train, a_train = _pack(train_idx)
    X_val, y_val, a_val = _pack(val_idx)
    X_test, y_test, a_test = _pack(test_idx)

    return TrainValTestSplit(
        X_train=X_train,
        X_val=X_val,
        X_test=X_test,
        y_train=y_train,
        y_val=y_val,
        y_test=y_test,
        acquisition_prob_train=a_train,
        acquisition_prob_val=a_val,
        acquisition_prob_test=a_test,
    )

split = temporal_or_random_split(
    df,
    train_fraction=float(CFG["data"]["train_fraction"]),
    val_fraction=float(CFG["data"]["val_fraction"]),
    random_seed=SEED,
)

print("train:", split.X_train.shape, "val:", split.X_val.shape, "test:", split.X_test.shape)

prep = make_preprocessor()
prep.fit(split.X_train, split.y_train)
feature_names = prep.get_feature_names_out()
print("num engineered features:", len(feature_names))
list(feature_names[:12])